In [0]:
%sql
-- LIST THE CUSTOMERS AND THEIR TOTAL QUANTITY OF PRODUCTS ORDERED.
with customer_order as (
select c.customer_id,c.first_name,sum(od.quantity) as total_quantity
from customers c 
join orders o
on c.customer_id=o.customer_id
join order_details od
on o.order_id=od.order_id
group by c.customer_id,c.first_name
)

select customer_id,first_name,total_quantity from customer_order ;

In [0]:
%sql
-- FIND THE PRODUCTS WITH THEIR TOTAL SALES (PRICE * QUANTITY )
with product_detail(
select product_name,
sum(quantity * price_each ) as total_sales,quantity
from order_details 
group by product_name,quantity
)

select product_name,total_sales,quantity from product_detail;

### MORE QUESTIONS 


In [0]:
%sql
-- 1. Calculate each customer's total spending.
with customer_spent as (
select 
c.customer_id,
concat(first_name,' ',last_name) as full_name,
coalesce(sum(od.quantity * od.price_each),0) as total_spent
from customers c 
left join orders o
on c.customer_id=o.customer_id
left join order_details od 
on o.order_id=od.order_id
group by c.customer_id,c.first_name,c.last_name
)
select customer_id,full_name,total_spent from customer_spent;

In [0]:
%sql
-- 2. Find each product's total quantity sold, showing only products whose total quantity is greater than 5.
with product_details as 
(
select product_name,sum(quantity) as total_sold 
from order_details 
group by product_name
)
select product_name,total_sold from product_details
where total_sold > 5; 

In [0]:
%sql
-- 3. Find customers whose total spending is greater than 1,000.
with customer_spent as 
(
select c.customer_id,
concat(c.first_name,' ',c.last_name) as full_name,
coalesce(sum(od.quantity*od.price_each),0) as total_spent 
from customers c 
left join orders o 
on c.customer_id=o.customer_id
left join order_details od 
on o.order_id=od.order_id
group by c.customer_id,c.first_name,c.last_name
)
select customer_id,full_name,total_spent from customer_spent 
where total_spent > 1000;

In [0]:
%sql
-- 4. Use a CTE to calculate each customer's total spending, then categorize them.
with total_spending as 
(
    select c.customer_id,
    concat(first_name,' ',last_name) as full_name,
    coalesce(sum(od.quantity*od.price_each),0) as total_spent 
    from customers c 
    left join orders o 
    on c.customer_id=o.customer_id
    left join order_details od 
    on o.order_id=od.order_id
    group by c.customer_id,c.first_name,c.last_name
)

select customer_id,full_name,total_spent, 
    case 
        when total_spent = 0 then 'No orders'
        when total_spent < 1000 then 'Bronze'
        when total_spent between 1000 and 3000 then 'Silver'
        else 'Gold'
    end as category_check
from total_spending;


In [0]:
%sql
-- 5. Find each customer's name in lowercase, total spending rounded to two decimal places, and order count. Include customers who have never placed an order.
with customer_details as 
(

    select lower(concat(c.first_name," ",c.last_name)) as full_name,
    coalesce(round(sum(od.quantity*od.price_each),2),0) as total_spent,
    count(distinct o.order_id) as order_count

    from customers c 
    left join orders o
    on c.customer_id=o.customer_id
    left join order_details od 
    on o.order_id=od.order_id
    group by c.customer_id,c.first_name,c.last_name
)
select full_name,total_spent,order_count from customer_details;
